# 06 — Classical Machine-Learning Baselines

Establish classical model baselines for the Paderborn feature and sequence experiments. Interpret the binary Healthy/Damaged task separately from the more difficult fault-location classification task.

In [ ]:
# ============================================================
# STAGE 6: CLASSICAL ML BASELINES
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROCESSED_DIR = Path("../data/processed")

feature_path = (
    PROCESSED_DIR /
    "paderborn_vibration_features.csv"
)

ranking_path = (
    PROCESSED_DIR /
    "training_feature_ranking.csv"
)

# ------------------------------------------------------------
# Load data
# ------------------------------------------------------------

df = pd.read_csv(feature_path)

feature_ranking = pd.read_csv(
    ranking_path
)

print("Dataset shape:", df.shape)

# ------------------------------------------------------------
# Recreate the FROZEN bearing-level split
# ------------------------------------------------------------

train_bearings = [
    "K001", "K002", "K003", "K004",
    "KA01", "KA04", "KA05", "KA06",
    "KA07", "KA09", "KA15", "KA16", "KA30",
    "KI01", "KI04", "KI05", "KI07",
    "KI08", "KI14", "KI16", "KI17",
    "KB23"
]

val_bearings = [
    "K005",
    "KA03", "KA08",
    "KI03",
    "KB24"
]

test_bearings = [
    "K006",
    "KA22",
    "KI18", "KI21",
    "KB27"
]

train_df = df[
    df["bearing"].isin(train_bearings)
].copy()

val_df = df[
    df["bearing"].isin(val_bearings)
].copy()

test_df = df[
    df["bearing"].isin(test_bearings)
].copy()

# ------------------------------------------------------------
# Feature sets
# ------------------------------------------------------------

top_12_features = (
    feature_ranking
    .head(12)["feature"]
    .tolist()
)

top_15_features = (
    feature_ranking
    .head(15)["feature"]
    .tolist()
)

all_22_features = [
    "mean",
    "std",
    "min",
    "max",
    "kurtosis",
    "skewness",
    "crest_factor",
    "dominant_frequency",
    "dominant_magnitude",
    "spectral_centroid",
    "spectral_bandwidth",
    "band_power_0_500",
    "band_power_500_1000",
    "band_power_1000_2000",
    "band_power_2000_3000",
    "band_power_3000_4000",
    "band_power_4000_5000",
    "band_power_5000_6000",
    "band_power_6000_7000",
    "band_power_7000_8000",
    "band_power_8000_9000",
    "band_power_9000_10000"
]

feature_sets = {
    "Top 12": top_12_features,
    "Top 15": top_15_features,
    "All 22": all_22_features
}

# ------------------------------------------------------------
# Basic verification
# ------------------------------------------------------------

print("\nRows:")
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\nBearings:")
print("Train:", train_df["bearing"].nunique())
print("Validation:", val_df["bearing"].nunique())
print("Test:", test_df["bearing"].nunique())

print("\nFeature counts:")
for name, features in feature_sets.items():
    print(f"{name}: {len(features)}")

In [ ]:
# ============================================================
# COMMON EVALUATION FUNCTION
# ============================================================

def evaluate_model(
    model,
    X_train,
    y_train,
    X_val,
    y_val,
    model_name,
    feature_set_name
):
    """
    Train on training bearings and evaluate on validation bearings.
    """

    model.fit(X_train, y_train)

    val_pred = model.predict(X_val)

    return {
        "model": model_name,
        "feature_set": feature_set_name,
        "accuracy": accuracy_score(y_val, val_pred),
        "balanced_accuracy": balanced_accuracy_score(
            y_val, val_pred
        ),
        "precision": precision_score(
            y_val, val_pred, zero_division=0
        ),
        "recall": recall_score(
            y_val, val_pred, zero_division=0
        ),
        "f1": f1_score(
            y_val, val_pred, zero_division=0
        ),
        "macro_f1": f1_score(
            y_val,
            val_pred,
            average="macro",
            zero_division=0
        ),
        "confusion_matrix": confusion_matrix(
            y_val,
            val_pred
        )
    }

In [ ]:
# ============================================================
# CLASSICAL ML MODELS
# ============================================================

models = {

    "Logistic Regression": Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]),

    "KNN": Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            KNeighborsClassifier(
                n_neighbors=5,
                weights="distance"
            )
        )
    ]),

    "SVM": Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            SVC(
                kernel="rbf",
                C=1.0,
                gamma="scale",
                class_weight="balanced",
                random_state=42
            )
        )
    ]),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    )
}

print("Models:")
for name in models:
    print("-", name)

In [ ]:
# ============================================================
# STAGE 6A: CLASSICAL BASELINE COMPARISON
# STAGE 1 - HEALTHY vs DAMAGED
# ============================================================

results = []

X_train_y = train_df["binary_label"]
X_val_y = val_df["binary_label"]

for feature_set_name, features in feature_sets.items():

    X_train = train_df[features]
    X_val = val_df[features]

    for model_name, model in models.items():

        print(
            f"Running: {model_name} | {feature_set_name}"
        )

        result = evaluate_model(
            model=model,
            X_train=X_train,
            y_train=X_train_y,
            X_val=X_val,
            y_val=X_val_y,
            model_name=model_name,
            feature_set_name=feature_set_name
        )

        results.append(result)

results_df = pd.DataFrame(results)

print("\nCompleted experiments:", len(results_df))

In [ ]:
# ============================================================
# VALIDATION RESULTS
# ============================================================

display(
    results_df[
        [
            "model",
            "feature_set",
            "accuracy",
            "balanced_accuracy",
            "precision",
            "recall",
            "f1",
            "macro_f1"
        ]
    ]
    .sort_values(
        ["feature_set", "balanced_accuracy"],
        ascending=[True, False]
    )
    .round(4)
)

In [ ]:
# ============================================================
# FEATURE-SET COMPARISON
# ============================================================

feature_comparison = (
    results_df
    .groupby("feature_set")[
        [
            "accuracy",
            "balanced_accuracy",
            "f1",
            "macro_f1"
        ]
    ]
    .mean()
    .sort_values(
        "balanced_accuracy",
        ascending=False
    )
)

display(
    feature_comparison.round(4)
)

## Stage 2 — Fault-Location Classification

This stage classifies damaged bearings into:

0 = outer-ring damage
1 = inner-ring damage
2 = combined inner + outer damage

In [ ]:
# ============================================================
# STAGE 2: OUTER vs INNER vs BOTH
# ============================================================

# Keep damaged measurements only
stage2_train_df = train_df[
    train_df["binary_label"] == 1
].copy()

stage2_val_df = val_df[
    val_df["binary_label"] == 1
].copy()

stage2_test_df = test_df[
    test_df["binary_label"] == 1
].copy()

# Original fault classes:
# 1 = outer
# 2 = inner
# 3 = both
#
# Remap for the classifier:
# 0 = outer
# 1 = inner
# 2 = both

stage2_label_map = {
    1: 0,
    2: 1,
    3: 2
}

stage2_train_df["stage2_label"] = (
    stage2_train_df["fault_class"]
    .map(stage2_label_map)
)

stage2_val_df["stage2_label"] = (
    stage2_val_df["fault_class"]
    .map(stage2_label_map)
)

stage2_test_df["stage2_label"] = (
    stage2_test_df["fault_class"]
    .map(stage2_label_map)
)

print("Stage 2 training distribution:")
print(
    stage2_train_df["stage2_label"]
    .value_counts()
    .sort_index()
)

print("\nStage 2 validation distribution:")
print(
    stage2_val_df["stage2_label"]
    .value_counts()
    .sort_index()
)

print("\nStage 2 test distribution:")
print(
    stage2_test_df["stage2_label"]
    .value_counts()
    .sort_index()
)

In [ ]:
# ============================================================
# STAGE 2 MODEL COMPARISON
# ============================================================

stage2_results = []

for feature_set_name, features in feature_sets.items():

    X_train = stage2_train_df[features]
    X_val = stage2_val_df[features]

    y_train = stage2_train_df["stage2_label"]
    y_val = stage2_val_df["stage2_label"]

    for model_name, model in models.items():

        print(
            f"Running: {model_name} | {feature_set_name}"
        )

        model.fit(X_train, y_train)

        val_pred = model.predict(X_val)

        result = {
            "model": model_name,
            "feature_set": feature_set_name,
            "accuracy": accuracy_score(
                y_val,
                val_pred
            ),
            "balanced_accuracy": balanced_accuracy_score(
                y_val,
                val_pred
            ),
            "macro_f1": f1_score(
                y_val,
                val_pred,
                average="macro",
                zero_division=0
            )
        }

        stage2_results.append(result)

stage2_results_df = pd.DataFrame(
    stage2_results
)

display(
    stage2_results_df.round(4)
)

In [ ]:
# ============================================================
# STAGE 2 CONFUSION MATRICES
# ============================================================

for _, row in stage2_results_df.iterrows():

    model_name = row["model"]
    feature_set_name = row["feature_set"]

    features = feature_sets[feature_set_name]

    X_train = stage2_train_df[features]
    X_val = stage2_val_df[features]

    y_train = stage2_train_df["stage2_label"]
    y_val = stage2_val_df["stage2_label"]

    # Recreate model so each evaluation is independent
    model = models[model_name]

    model.fit(X_train, y_train)

    val_pred = model.predict(X_val)

    print(
        f"\n{model_name} | {feature_set_name}"
    )

    print(
        confusion_matrix(
            y_val,
            val_pred,
            labels=[0, 1, 2]
        )
    )

In [ ]:
# ============================================================
# FINAL TEST - STAGE 1
# Healthy vs Damaged
# Selected feature set: All 22
# ============================================================

from sklearn.base import clone

selected_stage1_features = all_22_features

X_train = train_df[selected_stage1_features]
y_train = train_df["binary_label"]

X_test = test_df[selected_stage1_features]
y_test = test_df["binary_label"]

stage1_test_results = []

for model_name, model in models.items():

    final_model = clone(model)

    final_model.fit(X_train, y_train)

    test_pred = final_model.predict(X_test)

    result = {
        "model": model_name,
        "accuracy": accuracy_score(
            y_test, test_pred
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_test, test_pred
        ),
        "precision": precision_score(
            y_test, test_pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_test, test_pred,
            zero_division=0
        ),
        "f1": f1_score(
            y_test, test_pred,
            zero_division=0
        ),
        "macro_f1": f1_score(
            y_test, test_pred,
            average="macro",
            zero_division=0
        )
    }

    stage1_test_results.append(result)

    print(f"\n{model_name}")
    print(
        confusion_matrix(
            y_test,
            test_pred,
            labels=[0, 1]
        )
    )

stage1_test_results_df = pd.DataFrame(
    stage1_test_results
)

print("\nSTAGE 1 FINAL TEST RESULTS")
display(
    stage1_test_results_df.round(4)
)

In [ ]:
# ============================================================
# FINAL TEST - STAGE 2
# Outer vs Inner vs Both
# Selected feature set: Top 12
# ============================================================

selected_stage2_features = top_12_features

X_train = stage2_train_df[selected_stage2_features]
y_train = stage2_train_df["stage2_label"]

X_test = stage2_test_df[selected_stage2_features]
y_test = stage2_test_df["stage2_label"]

stage2_test_results = []

for model_name, model in models.items():

    final_model = clone(model)

    final_model.fit(X_train, y_train)

    test_pred = final_model.predict(X_test)

    result = {
        "model": model_name,
        "accuracy": accuracy_score(
            y_test, test_pred
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_test, test_pred
        ),
        "macro_f1": f1_score(
            y_test,
            test_pred,
            average="macro",
            zero_division=0
        )
    }

    stage2_test_results.append(result)

    print(f"\n{model_name}")
    print(
        confusion_matrix(
            y_test,
            test_pred,
            labels=[0, 1, 2]
        )
    )

stage2_test_results_df = pd.DataFrame(
    stage2_test_results
)

print("\nSTAGE 2 FINAL TEST RESULTS")
display(
    stage2_test_results_df.round(4)
)

In [ ]:
# ============================================================
# FINAL LEAKAGE / SPLIT AUDIT
# ============================================================

print("=" * 70)
print("DATA LEAKAGE AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Bearing overlap
# ------------------------------------------------------------

train_set = set(train_df["bearing"].unique())
val_set = set(val_df["bearing"].unique())
test_set = set(test_df["bearing"].unique())

print("\n1. BEARING OVERLAP")

print("Train ∩ Validation:", train_set & val_set)
print("Train ∩ Test      :", train_set & test_set)
print("Validation ∩ Test :", val_set & test_set)

# ------------------------------------------------------------
# 2. File overlap
# ------------------------------------------------------------

train_files = set(train_df["file"])
val_files = set(val_df["file"])
test_files = set(test_df["file"])

print("\n2. FILE OVERLAP")

print("Train ∩ Validation:", len(train_files & val_files))
print("Train ∩ Test      :", len(train_files & test_files))
print("Validation ∩ Test :", len(val_files & test_files))

# ------------------------------------------------------------
# 3. Target columns accidentally in feature sets
# ------------------------------------------------------------

target_columns = {
    "binary_label",
    "fault_class",
    "fault_location",
    "bearing",
    "file",
    "operating_condition",
    "measurement_number"
}

print("\n3. TARGET / ID FEATURES")

for set_name, features in feature_sets.items():

    leakage_features = set(features) & target_columns

    print(
        f"{set_name}:",
        leakage_features
        if leakage_features
        else "None"
    )

# ------------------------------------------------------------
# 4. Verify test bearings
# ------------------------------------------------------------

print("\n4. TEST BEARINGS")

print(
    sorted(test_set)
)

print(
    "\nMeasurements per test bearing:"
)

print(
    test_df.groupby("bearing").size()
)

# ------------------------------------------------------------
# 5. Verify target labels are correct
# ------------------------------------------------------------

print("\n5. TEST LABEL DISTRIBUTION")

print(
    test_df[
        ["bearing", "fault_location", "binary_label"]
    ]
    .drop_duplicates()
    .sort_values("bearing")
)

# ------------------------------------------------------------
# 6. Verify feature-selection source
# ------------------------------------------------------------

print("\n6. SELECTED STAGE-1 FEATURES")

print(
    "Stage-1 selected feature set:",
    "All 22"
)

print(
    "ANOVA ranking generated from:",
    "train_df only"
)

print(
    "\nLeakage audit completed."
)

In [ ]:
# ============================================================
# PER-BEARING FINAL TEST CHECK - STAGE 1
# ============================================================

from sklearn.base import clone

features = all_22_features

X_train = train_df[features]
y_train = train_df["binary_label"]

X_test = test_df[features]
y_test = test_df["binary_label"]

test_bearing_results = []

for model_name, original_model in models.items():

    model = clone(original_model)

    model.fit(X_train, y_train)

    test_pred = model.predict(X_test)

    temp = test_df[
        ["bearing", "fault_location"]
    ].copy()

    temp["true_label"] = y_test.to_numpy()
    temp["predicted_label"] = test_pred

    for bearing, group in temp.groupby("bearing"):

        test_bearing_results.append({
            "model": model_name,
            "bearing": bearing,
            "fault_location": group[
                "fault_location"
            ].iloc[0],
            "samples": len(group),
            "correct": (
                group["true_label"]
                == group["predicted_label"]
            ).sum(),
            "accuracy": (
                group["true_label"]
                == group["predicted_label"]
            ).mean()
        })

per_bearing_df = pd.DataFrame(
    test_bearing_results
)

display(
    per_bearing_df.round(4)
)

In [ ]:
# ============================================================
# PER-OPERATING-CONDITION FINAL TEST CHECK
# ============================================================

condition_results = []

for model_name, original_model in models.items():

    model = clone(original_model)

    model.fit(
        train_df[features],
        train_df["binary_label"]
    )

    test_pred = model.predict(
        test_df[features]
    )

    temp = test_df[
        [
            "bearing",
            "operating_condition",
            "binary_label"
        ]
    ].copy()

    temp["prediction"] = test_pred

    for (bearing, condition), group in temp.groupby(
        ["bearing", "operating_condition"]
    ):

        condition_results.append({
            "model": model_name,
            "bearing": bearing,
            "condition": condition,
            "samples": len(group),
            "accuracy": (
                group["binary_label"]
                == group["prediction"]
            ).mean()
        })

per_condition_df = pd.DataFrame(
    condition_results
)

display(
    per_condition_df.round(4)
)

In [ ]:
# ============================================================
# EXACT / NEAR-DUPLICATE FEATURE AUDIT
# ============================================================

features = all_22_features

X_train_audit = train_df[features].to_numpy()
X_test_audit = test_df[features].to_numpy()

# Exact duplicate rows
train_rows = {
    tuple(row)
    for row in X_train_audit
}

exact_matches = sum(
    tuple(row) in train_rows
    for row in X_test_audit
)

print("Exact test rows matching a training row:",
      exact_matches)

# Nearest training distance after per-feature standardization
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances

audit_scaler = StandardScaler()

X_train_scaled = audit_scaler.fit_transform(
    X_train_audit
)

X_test_scaled = audit_scaler.transform(
    X_test_audit
)

distances = pairwise_distances(
    X_test_scaled,
    X_train_scaled,
    metric="euclidean"
)

nearest_distances = distances.min(axis=1)

print("\nNearest training distance:")
print("Minimum :", nearest_distances.min())
print("Median  :", np.median(nearest_distances))
print("Maximum :", nearest_distances.max())

print(
    "\nTest rows with nearest training distance < 0.01:",
    np.sum(nearest_distances < 0.01)
)

In [ ]:
# ============================================================
# VISUALIZE KNN RESULT USING PCA
# IMPORTANT:
# PCA is fitted ONLY on training data.
# This is a visualization, not the actual KNN space.
# ============================================================

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

features = all_22_features

# ------------------------------------------------------------
# Prepare data
# ------------------------------------------------------------

X_train = train_df[features].to_numpy()
X_test = test_df[features].to_numpy()

y_train = train_df["binary_label"].to_numpy()
y_test = test_df["binary_label"].to_numpy()

# ------------------------------------------------------------
# Scale using training data only
# ------------------------------------------------------------

viz_scaler = StandardScaler()

X_train_scaled = viz_scaler.fit_transform(X_train)
X_test_scaled = viz_scaler.transform(X_test)

# ------------------------------------------------------------
# PCA fitted ONLY on training data
# ------------------------------------------------------------

pca = PCA(n_components=2)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(
    "Variance explained by PC1:",
    round(pca.explained_variance_ratio_[0], 4)
)

print(
    "Variance explained by PC2:",
    round(pca.explained_variance_ratio_[1], 4)
)

print(
    "Total variance explained:",
    round(
        pca.explained_variance_ratio_.sum(),
        4
    )
)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

plt.figure(figsize=(11, 8))

# Training points
plt.scatter(
    X_train_pca[y_train == 0, 0],
    X_train_pca[y_train == 0, 1],
    alpha=0.35,
    label="Training - Healthy"
)

plt.scatter(
    X_train_pca[y_train == 1, 0],
    X_train_pca[y_train == 1, 1],
    alpha=0.35,
    label="Training - Damaged"
)

# Test points
plt.scatter(
    X_test_pca[y_test == 0, 0],
    X_test_pca[y_test == 0, 1],
    marker="x",
    s=70,
    label="Test - Healthy"
)

plt.scatter(
    X_test_pca[y_test == 1, 0],
    X_test_pca[y_test == 1, 1],
    marker="x",
    s=70,
    label="Test - Damaged"
)

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title(
    "PCA Visualization of KNN Healthy vs Damaged Data"
)

plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# KNN NEAREST-NEIGHBOR DISTANCE VISUALIZATION
# ============================================================

from sklearn.neighbors import NearestNeighbors

# Use the same training-only scaling
nn = NearestNeighbors(n_neighbors=5)

nn.fit(X_train_scaled)

distances, indices = nn.kneighbors(
    X_test_scaled
)

# Distance to the closest training sample
nearest_distance = distances[:, 0]

# True test classes
healthy_distances = nearest_distance[
    y_test == 0
]

damaged_distances = nearest_distance[
    y_test == 1
]

plt.figure(figsize=(11, 6))

plt.hist(
    healthy_distances,
    bins=30,
    alpha=0.6,
    label="Healthy test measurements"
)

plt.hist(
    damaged_distances,
    bins=30,
    alpha=0.6,
    label="Damaged test measurements"
)

plt.xlabel(
    "Distance to nearest training measurement"
)

plt.ylabel("Number of test measurements")

plt.title(
    "Nearest Training-Point Distance for KNN Test Measurements"
)

plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# TEST-BEARING KNN VISUALIZATION
# ============================================================

# Refit the actual KNN pipeline used in our experiment
knn_model = models["KNN"]

knn_model.fit(
    train_df[features],
    train_df["binary_label"]
)

test_predictions = knn_model.predict(
    test_df[features]
)

plot_df = test_df[
    ["bearing", "fault_location", "binary_label"]
].copy()

plot_df["prediction"] = test_predictions

bearing_accuracy = (
    plot_df
    .groupby(
        ["bearing", "fault_location"]
    )
    .apply(
        lambda g:
        (g["binary_label"] == g["prediction"]).mean()
    )
    .reset_index(
        name="accuracy"
    )
)

display(bearing_accuracy)